In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Feature Engineering for DDoS Detection\n",
    "\n",
    "This notebook focuses on creating advanced features from raw network flow data to improve machine learning model performance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import math\n",
    "from collections import defaultdict\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy import stats\n",
    "\n",
    "# Load raw flow data (example)\n",
    "# flows = pd.read_parquet('data/raw/flows.parquet')\n",
    "\n",
    "# Generate synthetic data for demonstration\n",
    "np.random.seed(42)\n",
    "n_flows = 50000\n",
    "flows = pd.DataFrame({\n",
    "    'timestamp': np.linspace(0, 3600, n_flows),  # 1 hour\n",
    "    'src_ip': np.random.choice(['10.0.0.{}'.format(i) for i in range(1, 101)], n_flows),\n",
    "    'dst_ip': np.random.choice(['192.168.1.{}'.format(i) for i in range(1, 51)], n_flows),\n",
    "    'src_port': np.random.choice([80, 443, 53, 12345, 54321], n_flows),\n",
    "    'dst_port': np.random.choice([80, 443, 53, 22, 3389], n_flows),\n",
    "    'protocol': np.random.choice([6, 17, 1], n_flows, p=[0.6, 0.3, 0.1]),\n",
    "    'bytes': np.random.exponential(500, n_flows),\n",
    "    'packets': np.random.poisson(10, n_flows),\n",
    "    'duration': np.random.exponential(0.5, n_flows),\n",
    "    'tcp_flags': np.random.choice([0, 2, 4, 16, 18], n_flows, p=[0.7, 0.1, 0.05, 0.1, 0.05])\n",
    "})\n",
    "print(f\"Number of flows: {len(flows)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Function to compute entropy\n",
    "def entropy(series):\n",
    "    counts = series.value_counts()\n",
    "    probs = counts / len(series)\n",
    "    return -np.sum(probs * np.log2(probs))\n",
    "\n",
    "# Create time windows (e.g., 10-second windows)\n",
    "window_size = 10  # seconds\n",
    "flows['time_window'] = (flows['timestamp'] // window_size).astype(int)\n",
    "\n",
    "# Aggregate features per window\n",
    "window_features = flows.groupby('time_window').agg({\n",
    "    'bytes': ['sum', 'mean', 'std'],\n",
    "    'packets': ['sum', 'mean', 'std'],\n",
    "    'duration': ['mean', 'std'],\n",
    "    'src_ip': lambda x: entropy(x),\n",
    "    'dst_ip': lambda x: entropy(x),\n",
    "    'src_port': lambda x: entropy(x),\n",
    "    'dst_port': lambda x: entropy(x),\n",
    "    'protocol': lambda x: entropy(x),\n",
    "    'tcp_flags': lambda x: entropy(x),\n",
    "}).reset_index()\n",
    "\n",
    "# Flatten column names\n",
    "window_features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in window_features.columns.values]\n",
    "window_features.rename(columns={'time_window_': 'time_window'}, inplace=True)\n",
    "print(f\"Window features shape: {window_features.shape}\")\n",
    "window_features.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Additional features: packet size statistics\n",
    "flows['packet_size'] = flows['bytes'] / flows['packets']\n",
    "flows['small_packet'] = (flows['packet_size'] < 64).astype(int)\n",
    "\n",
    "# Per window: small packet ratio, average packet size\n",
    "size_features = flows.groupby('time_window').agg({\n",
    "    'packet_size': ['mean', 'std'],\n",
    "    'small_packet': 'mean'\n",
    "}).reset_index()\n",
    "size_features.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in size_features.columns.values]\n",
    "window_features = window_features.merge(size_features, on='time_window', how='left')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# TCP flag ratios per window\n",
    "def syn_ratio(group):\n",
    "    return (group['tcp_flags'] & 2 > 0).mean()\n",
    "\n",
    "def rst_ratio(group):\n",
    "    return (group['tcp_flags'] & 4 > 0).mean()\n",
    "\n",
    "def fin_ratio(group):\n",
    "    return (group['tcp_flags'] & 1 > 0).mean()\n",
    "\n",
    "def ack_ratio(group):\n",
    "    return (group['tcp_flags'] & 16 > 0).mean()\n",
    "\n",
    "flag_features = flows.groupby('time_window').apply(\n",
    "    lambda x: pd.Series({\n",
    "        'syn_ratio': syn_ratio(x),\n",
    "        'rst_ratio': rst_ratio(x),\n",
    "        'fin_ratio': fin_ratio(x),\n",
    "        'ack_ratio': ack_ratio(x)\n",
    "    })\n",
    ").reset_index()\n",
    "\n",
    "window_features = window_features.merge(flag_features, on='time_window', how='left')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Handle missing values (fill NaN with 0)\n",
    "window_features = window_features.fillna(0)\n",
    "\n",
    "# Add attack label if available (simulate)\n",
    "# In real data, you would merge with ground truth\n",
    "np.random.seed(42)\n",
    "window_features['attack'] = np.random.choice([0, 1], size=len(window_features), p=[0.95, 0.05])\n",
    "\n",
    "print(\"Final feature set shape:\", window_features.shape)\n",
    "window_features.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation with attack label\n",
    "corr_with_attack = window_features.corr()['attack'].sort_values(ascending=False)\n",
    "plt.figure(figsize=(10,6))\n",
    "corr_with_attack.drop('attack').plot(kind='barh')\n",
    "plt.title('Feature Correlation with Attack Label')\n",
    "plt.tight_layout()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save engineered features for model training\n",
    "# window_features.to_parquet('data/processed/engineered_features.parquet', index=False)\n",
    "print(\"Features saved.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}